# Feature engineering
- Localization (Cambodia context)
- Creation
- Transforming

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/processed/cleaned_credit_risk_dataset.csv')

## LOCALIZATION (CAMBODIA CONTEXT)

Mapping Global Housing to Cambodian Land Titles
- Rent -> Soft Title, Mortgage/Own -> Hard Title, Other -> No Title

In [3]:
house_map = {
    'RENT': 'Soft Title',
    'MORTGAGE': 'Hard Title',
    'OWN': 'Hard Title',
    'OTHER': 'No Title'
}
df['land_title_type'] = df['person_home_ownership'].map(house_map)

Re-scaling Income to Cambodian Context ($200 - $1,500/month)
- use a Min-Max scaling approach to fit the original distribution into the $2,400 - $18,000 yearly range
- by converting each value into a percentage of its original spread and then "stretching" it to fit the new cam economic scale

In [4]:
min_cam_inc = 2400
max_cam_inc = 18000
df['person_income_kh'] = ((df['person_income'] - df['person_income'].min()) / 
                         (df['person_income'].max() - df['person_income'].min()) * (max_cam_inc - min_cam_inc) + min_cam_inc)

Mapping Global Loan Grade to NBC Status (Khmer)
- A / B -> Normal (Standard)
- C -> Special Mention
- D -> Substandard
- E -> Doubtful
- F / G -> Loss

In [5]:
grade_map = {
    'A': 'Normal (Standard)',
    'B': 'Normal (Standard)',
    'C': 'Special Mention',
    'D': 'Substandard',
    'E': 'Doubtful',
    'F': 'Loss',
    'G': 'Loss'
}
df['nbc_status'] = df['loan_grade'].map(grade_map)

## FEATURE CREATION (PROXY INDICATORS)

Debt-to-Income Ratio (Already exists as loan_percent_income, but let's ensure it's clear)
- Higher ratio = Higher risk for informal workers

In [6]:
df['loan_to_income_ratio'] = df['loan_amnt'] / df['person_income']

Financial Stability Index 
- Combines Employment Length and Credit History Length

In [7]:
df['stability_score'] = df['person_emp_length'] + df['cb_person_cred_hist_length']

Interest Burden
- How much interest is the person paying relative to their income?

In [8]:
df['interest_burden'] = (df['loan_amnt'] * (df['loan_int_rate']/100)) / df['person_income']

## CATEGORICAL ENCODING

One-Hot Encoding for Loan Intent and Land Title
- This creates binary columns (0 or 1) for each category

In [9]:
df = pd.get_dummies(df, columns=['loan_intent', 'land_title_type', 'cb_person_default_on_file', 'nbc_status'], drop_first=True)

Drop original categorical columns that are no longer needed

In [10]:
df = df.drop(['person_home_ownership', 'loan_grade'], axis=1)

## RESULT

In [11]:
print(f"New shape of dataset: {df.shape}")
print(df.head())

New shape of dataset: (31650, 24)
   person_age  person_income  person_emp_length  loan_amnt  loan_int_rate  \
0          21           9600                5.0       1000          11.14   
1          25           9600                1.0       5500          12.87   
2          23          65500                4.0      35000          15.23   
3          24          54400                8.0      35000          14.27   
4          21           9900                2.0       2500           7.14   

   loan_status  loan_percent_income  cb_person_cred_hist_length  \
0            0                 0.10                           2   
1            1                 0.57                           3   
2            1                 0.53                           2   
3            1                 0.55                           4   
4            1                 0.25                           2   

   person_income_kh  loan_to_income_ratio  ...  loan_intent_MEDICAL  \
0       2442.912215          

In [12]:
df.to_csv('../data/processed/featured_credit_risk_kh.csv', index=False)